# 2 層樓 8 柱 RC 構架耐震設計——逐課計算 Notebook

依據:《建築物耐震設計規範及解說》113 年 3 月 1 日修正版

每一個 Section 對應規範一個章節,查表值與公式編號都標明出處。
本 Notebook 是講義的可執行版本——改上面的輸入參數,下面所有結果會跟著重算。

In [ ]:
import math

# ============================================================
# 課前:設計輸入(不是規範查表值,是本案例的假設條件)
# ============================================================
site_city   = "桃園市"
site_dist   = "桃園區"
usage       = "一般辦公"
struct_sys  = "RC 特殊抗彎矩構架 (SMF)"

n_story     = 2
story_h     = 3.5      # m, 每層淨高
hn          = n_story * story_h   # 總高度(m)

bay_x, n_bay_x = 6.0, 3   # X向: 3跨@6m = 18m
bay_y, n_bay_y = 6.0, 1   # Y向: 1跨@6m = 6m
Lx = bay_x * n_bay_x
Ly = bay_y * n_bay_y
n_columns = (n_bay_x+1) * (n_bay_y+1)

soil_class  = "第二類地盤(假設,無鑽探資料)"

D_floor = 5.0   # kN/m^2, 樓板靜載重(含結構自重+面層+隔間)
D_roof  = 4.0   # kN/m^2, 屋頂靜載重

print(f"基地: {site_city}{site_dist}  結構系統: {struct_sys}")
print(f"總高度 hn = {hn} m,  平面 {Lx}m x {Ly}m,  柱數 = {n_columns}")

## 第 2 課:分析方法適用性(規範第 2 章開頭)

靜力分析法適用於「高度未達五十公尺或未達十五層之規則性建築物」。

In [ ]:
assert hn < 50 and n_story < 15, "超出靜力分析適用範圍,須改用動力分析"
print(f"hn={hn}m < 50m,  樓層數={n_story} < 15層  ->  等值靜力法適用")

## 第 3 課:地震危害度參數查表(規範表 2-1)

查「桃園市 / 桃園區」列。**這組數字必須依實際基地所在鄉鎮市區重新查表**,
不同區的數值差異很大(例如龜山區、大溪區數字就不一樣)。

In [ ]:
# 表2-1 查表值 (桃園市/桃園區)
SsD, S1D = 0.50, 0.30   # 設計地震
SsM, S1M = 0.80, 0.40   # 最大考量地震

print(f"SsD={SsD}g  S1D={S1D}g   (設計地震)")
print(f"SsM={SsM}g  S1M={S1M}g   (最大考量地震)")

## 第 4 課:工址土壤分類與放大係數(規範 2.4 節、表 2-4a/b)

VS30 判定第一/二/三類地盤。本案例假設**第二類地盤**(無鑽探資料時的常見保守假設,
正式設計仍須依規範第一章第二節辦理地基調查取得實際 VS30)。

In [ ]:
# 表2-4(a)/(b) 第二類地盤, 線性內插查值
Fa_D, Fv_D = 1.1, 1.5   # 對應 SsD=0.50, S1D=0.30
Fa_M, Fv_M = 1.0, 1.3   # 對應 SsM=0.80, S1M=0.40

SDS = Fa_D * SsD
SD1 = Fv_D * S1D
SMS = Fa_M * SsM
SM1 = Fv_M * S1M

print(f"SDS = Fa*SsD = {Fa_D}*{SsD} = {SDS:.3f}g")
print(f"SD1 = Fv*S1D = {Fv_D}*{S1D} = {SD1:.3f}g")
print(f"SMS = Fa*SsM = {Fa_M}*{SsM} = {SMS:.3f}g")
print(f"SM1 = Fv*S1M = {Fv_M}*{S1M} = {SM1:.3f}g")

## 第 5 課:用途係數(規範第 2.8 節)

一般辦公,非人群聚集、非重要建築物 -> 第四類建築物。

In [ ]:
I = 1.0
print(f"用途係數 I = {I}")

## 第 6 課:自然振動週期(規範式 2-8)

RC 構造(無剛性非結構牆/剪力牆/斜撐)經驗公式: T = 0.070 * hn^0.75

In [ ]:
Ct = 0.070
T = Ct * hn**0.75
print(f"T = {Ct} * {hn}^0.75 = {T:.4f} s")

## 第 7 課:設計反應譜加速度係數 SaD(規範表 2-5a)

先算短週期/中週期分界點 T0D = SD1/SDS,再判斷 T 落在哪一段。

In [ ]:
T0D = SD1 / SDS
print(f"T0D = SD1/SDS = {SD1}/{SDS:.3f} = {T0D:.4f} s")
print(f"分界: 0.2T0D={0.2*T0D:.4f}s   0.6T0D={0.6*T0D:.4f}s   T0D={T0D:.4f}s")

def SaD_func(T, SDS, SD1, T0D):
    if T <= 0.2*T0D:
        return SDS*(0.4+3*T/T0D), "較短週期"
    elif T <= T0D:
        return SDS, "短週期"
    elif T <= 2.5*T0D:
        return SD1/T, "中週期"
    else:
        return 0.4*SDS, "長週期"

SaD, branch = SaD_func(T, SDS, SD1, T0D)
print(f"T={T:.4f}s 落於「{branch}」段  ->  SaD = {SaD:.4f}g")

## 第 8 課:韌性容量與容許韌性容量(規範表 1-3、式 2-10)

查表 1-3:RC 特殊抗彎矩構架 -> R=4.8,高度限制「不限」。

In [ ]:
R = 4.8   # 表1-3查值: RC特殊抗彎矩構架
Ra = 1 + (R-1)/1.5   # 式2-10, 一般工址(非台北盆地)
print(f"R = {R}")
print(f"Ra = 1+(R-1)/1.5 = {Ra:.4f}")

## 第 9 課:結構系統地震力折減係數 Fu(規範式 2-12)

Fu 依 T/T0D 的位置分四段定義。

In [ ]:
def Fu_func(T, T0D, Ra):
    sq = math.sqrt(2*Ra-1)
    if T >= T0D:
        return Ra, "T>=T0D"
    elif T >= 0.6*T0D:
        return sq + (Ra-sq)*(T-0.6*T0D)/(0.4*T0D), "0.6T0D<=T<T0D"
    elif T >= 0.2*T0D:
        return sq, "0.2T0D<=T<0.6T0D"
    else:
        return 1 + (sq-1)*T/(0.2*T0D), "T<0.2T0D"

Fu, Fu_branch = Fu_func(T, T0D, Ra)
print(f"T 落於「{Fu_branch}」段  ->  Fu = {Fu:.4f}")

## 第 10 課:折減比值修正(規範式 2-2)

In [ ]:
ratio = SaD / Fu
if ratio <= 0.3:
    ratio_mod = ratio
    branch2 = "第一段(不修正)"
elif ratio <= 0.8:
    ratio_mod = 0.52*ratio + 0.144
    branch2 = "第二段"
else:
    ratio_mod = 0.70*ratio
    branch2 = "第三段"

print(f"SaD/Fu = {ratio:.4f}  ->  {branch2}")
print(f"(SaD/Fu)' = {ratio_mod:.4f}")

## 第 11 課:起始降伏地震力放大倍數 γy(規範 2.9 節解說)

RC 構造採極限強度設計法: γy = 1.5

In [ ]:
gamma_y = 1.5
print(f"gamma_y = {gamma_y}")

## 第 12 課:重力載重與有效地震重量 W(規範第 2.2 節)

In [ ]:
area = Lx * Ly
W1 = area * D_floor   # 2F樓板
W2 = area * D_roof    # 屋頂
W  = W1 + W2

print(f"樓地板面積 = {Lx}m x {Ly}m = {area} m^2")
print(f"W1(2F)   = {area}*{D_floor} = {W1:.1f} kN")
print(f"W2(屋頂) = {area}*{D_roof} = {W2:.1f} kN")
print(f"W(合計)  = {W:.1f} kN")

## 第 13 課:最小設計水平總橫力 V(規範式 2-1)——關鍵結果

In [ ]:
V = (I * ratio_mod) / (1.4 * gamma_y) * W
print(f"V = [I*(SaD/Fu)'] / (1.4*gamma_y) * W")
print(f"  = [{I}*{ratio_mod:.4f}] / (1.4*{gamma_y}) * {W:.1f}")
print(f"  = {V:.2f} kN")

## 第 14 課:地震力垂直分配(規範式 2-14、2-15)

In [ ]:
Ft = 0.07*T*V if T > 0.7 else 0.0
Ft = min(Ft, 0.25*V)
print(f"T={T:.3f}s  ->  Ft = {Ft:.2f} kN  (T<=0.7s 時可為零)")

h1, h2 = story_h, hn
Wh1, Wh2 = W1*h1, W2*h2
sumWh = Wh1 + Wh2

F1 = (V-Ft)*Wh1/sumWh
F2 = (V-Ft)*Wh2/sumWh + Ft

print(f"F1(2F)   = {F1:.2f} kN")
print(f"F2(屋頂) = {F2:.2f} kN")
print(f"合計檢核 = {F1+F2:.2f} kN  (應等於 V={V:.2f} kN)")

## 第 15 課:各柱剪力與柱端彎矩概估(反曲點法,非規範條文,常用簡化分析)

假設 8 柱等剛度均分——正式設計仍須用矩陣分析或有限元素模型取代此假設。

In [ ]:
V_1F = V          # 1F(基底)層剪力
V_2F = F2          # 2F層剪力(=屋頂分配到的力)

Vcol_1F = V_1F / n_columns
Vcol_2F = V_2F / n_columns

M_1F = Vcol_1F * story_h / 2   # 反曲點法: M = V*h/2
M_2F = Vcol_2F * story_h / 2

print(f"1F 每柱剪力 = {Vcol_1F:.2f} kN,  柱端彎矩 = {M_1F:.2f} kN-m")
print(f"2F 每柱剪力 = {Vcol_2F:.2f} kN,  柱端彎矩 = {M_2F:.2f} kN-m")

## 第 16 課:容許層間相對側向位移角(規範 2.16.1 節)——重要提醒

規範明文:**不得超過 0.005**。此限制無法用本 notebook 目前的反曲點法算出實際位移
(反曲點法只給彎矩概估,位移誤差大),需要實際結構勁度矩陣。

**強烈建議**:把這裡算出的柱斷面初步結果,拿到 OpenSeesPy 建立實際 2 層構架模型,
跑靜力分析直接讀層間位移角,和這裡的簡化結果交叉驗證。

In [ ]:
drift_limit = 0.005
print(f"容許層間位移角 = {drift_limit}  (規範2.16.1節)")
print("此值無法由反曲點法可靠算出,建議用OpenSeesPy等有限元素工具驗證")

## 總結表

In [ ]:
print("="*50)
print("計算結果總結")
print("="*50)
print(f"{'SDS':<20}{SDS:.3f} g")
print(f"{'SD1':<20}{SD1:.3f} g")
print(f"{'自然週期 T':<20}{T:.4f} s")
print(f"{'Ra':<20}{Ra:.4f}")
print(f"{'Fu':<20}{Fu:.4f}")
print(f"{'基底剪力 V':<20}{V:.2f} kN")
print(f"{'2F層剪力 F1':<20}{F1:.2f} kN")
print(f"{'屋頂層剪力 F2':<20}{F2:.2f} kN")
print(f"{'1F柱端彎矩':<20}{M_1F:.2f} kN-m")
print(f"{'2F柱端彎矩':<20}{M_2F:.2f} kN-m")
print(f"{'容許層間位移角':<20}{drift_limit}")